## About the NDVI data

**What it is.** NDVI (Normalized Difference Vegetation Index) measures how green and
dense the vegetation is. It is calculated from the red and near-infrared light that
the surface reflects: healthy plants absorb red light and reflect near-infrared, so
they give high values. Values run from about -1 to 1:
- below ~0.2: bare soil, stubble, buildings
- ~0.2 to 0.5: sparse or young vegetation
- above ~0.5: dense, actively growing vegetation

**Source.** NASA MODIS product MOD13Q1 (version 061), downloaded through AppEEARS.
It has a 250 m pixel and one composite every 16 days since February 2000. Each value
is the best observation within a 16-day window, which reduces the effect of clouds.
Every observation comes with a `pixel_reliability` flag (0 = good, 1 = marginal,
3 = cloudy). Cloudy rows are removed before analysis.

**Why we use it.** NDVI shows how the crop in each zone actually behaved, which is the
"effect" in our chain of climate, water, crop stress, soil, and rotation. It lets us:
- identify the **crop calendar** (when greening starts, peaks, and ends)
- find **bare-soil periods**, when the field is exposed to erosion and water loss
- detect **stress years**, when the peak is lower than usual
- compare NDVI with rainfall and soil moisture to see how sensitive each zone is to dry conditions

**What we found for Écija.** NDVI rises through winter, peaks in March (average about
0.58), and falls to about 0.2 in summer, which matches a rainfed winter cereal.

**Limitations.** A 250 m pixel can mix different surfaces, so the value is an average
of the area, not of a single plant. A low yearly peak could mean drought, but also a
different crop or fallow that year, so it needs to be checked against rainfall and
soil moisture.

### 1. Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### 2. Functions

In [2]:
def quality_report(df):
    report = {}
    """Return a dictionary of quality checks for a DataFrame."""
    report['shape'] = df.shape
    report['columns'] = df.columns
    
    df = df.drop_duplicates()
    report['NaN_values'] = df.isna().sum()
    constant = [c for c in df.columns if df[c].nunique() == 1]
    report['constant_values'] = constant
    return df, report
    

In [3]:
def final_df(df):
    keep_final_columns = ['Date', 'MOD13Q1_061__250m_16_days_EVI','MOD13Q1_061__250m_16_days_NDVI','MOD13Q1_061__250m_16_days_pixel_reliability']
    df_processed = df[keep_final_columns]
    return df_processed

### 3. Configuration

In [4]:
# Paths to raw data
ndvi_raw_folder_loc = '../data/raw/ndvi/'

ecija_raw_file = 'ecija_ndvi.csv'
guadix_raw_file = 'guadix_ndvi.csv'
jerez_raw_file = 'jerez_ndvi.csv'
ubeda_raw_file = 'ubeda_ndvi.csv'

# Paths to processed data

ndvi_processed_folder_loc = '../data/processed/ndvi/'
os.makedirs(ndvi_processed_folder_loc, exist_ok=True)

ecija_processed_file = 'ecija_ndvi_p.csv'
guadix_processed_file = 'guadix_ndvi_p.csv'
jerez_processed_file = 'jerez_ndvi_p.csv'
ubeda_processed_file = 'ubeda_ndvi_p.csv'

ecija_raw_file_loc = os.path.join(ndvi_raw_folder_loc,ecija_raw_file)
guadix_raw_file_loc = os.path.join(ndvi_raw_folder_loc,guadix_raw_file)
jerez_raw_file_loc = os.path.join(ndvi_raw_folder_loc,jerez_raw_file)
ubeda_raw_file_loc = os.path.join(ndvi_raw_folder_loc,ubeda_raw_file)

### 4. Work with NDVI datasets

#### 4.1 Écija dataset

In [5]:
ecija_raw_df = pd.read_csv(ecija_raw_file_loc, parse_dates=["Date"])

In [6]:
ecija_raw_df.head()

,Category,ID,Latitude,Longitude,Date,MODIS_Tile,MOD13Q1_061_Line_Y_250m,MOD13Q1_061_Sample_X_250m,MOD13Q1_061__250m_16_days_EVI,MOD13Q1_061__250m_16_days_NDVI,...,MOD13Q1_061__250m_16_days_VI_Quality_Mixed_Clouds_Description,MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask,MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask_Description,MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice,MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice_Description,MOD13Q1_061__250m_16_days_VI_Quality_Possible_shadow,MOD13Q1_061__250m_16_days_VI_Quality_Possible_shadow_Description,MOD13Q1_061__250m_16_days_pixel_reliability_bitmask,MOD13Q1_061__250m_16_days_pixel_reliability_MODLAND,MOD13Q1_061__250m_16_days_pixel_reliability_MODLAND_Description
0,zone,ecija,37.55537,-5.14244,2000-02-18,h17v05,1173.0,2843.0,0.3269,0.4989,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
1,zone,ecija,37.55537,-5.14244,2000-03-05,h17v05,1173.0,2843.0,0.4503,0.5964,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
2,zone,ecija,37.55537,-5.14244,2000-03-21,h17v05,1173.0,2843.0,0.4772,0.6279,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
3,zone,ecija,37.55537,-5.14244,2000-04-06,h17v05,1173.0,2843.0,0.2790,0.5747,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
4,zone,ecija,37.55537,-5.14244,2000-04-22,h17v05,1173.0,2843.0,0.2579,0.4957,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"


In [7]:
ecija_raw_df.describe()

,Latitude,Longitude,Date,MOD13Q1_061_Line_Y_250m,MOD13Q1_061_Sample_X_250m,MOD13Q1_061__250m_16_days_EVI,MOD13Q1_061__250m_16_days_NDVI,MOD13Q1_061__250m_16_days_VI_Quality,MOD13Q1_061__250m_16_days_pixel_reliability
count,614.00000,6.140000e+02,614,614.0,614.0,614.000000,614.000000,614.000000,614.000000
mean,37.55537,-5.142440e+00,2013-06-17 16:32:03.127035,1173.0,2843.0,0.227895,0.351528,2236.618893,0.105863
min,37.55537,-5.142440e+00,2000-02-18 00:00:00,1173.0,2843.0,0.082000,0.155900,2062.000000,0.000000
25%,37.55537,-5.142440e+00,2006-10-20 00:00:00,1173.0,2843.0,0.126850,0.214925,2116.000000,0.000000
50%,37.55537,-5.142440e+00,2013-06-18 00:00:00,1173.0,2843.0,0.158500,0.264550,2116.000000,0.000000
75%,37.55537,-5.142440e+00,2020-02-14 00:00:00,1173.0,2843.0,0.271625,0.438875,2116.000000,0.000000
max,37.55537,-5.142440e+00,2026-10-16 00:00:00,1173.0,2843.0,0.865500,0.896600,35156.000000,3.000000
std,0.00000,8.889026e-16,NaN,0.0,0.0,0.155809,0.190298,1878.227597,0.391839


Columns with a single value in every row carry no information, because they can’t explain any variation in the data. They are checked and removed to keep the table smaller and easier to read.

In [8]:
ecija_constant = [c for c in ecija_raw_df.columns if ecija_raw_df[c].nunique() == 1]
print(ecija_constant)

['Category', 'ID', 'Latitude', 'Longitude', 'MODIS_Tile', 'MOD13Q1_061_Line_Y_250m', 'MOD13Q1_061_Sample_X_250m', 'MOD13Q1_061__250m_16_days_VI_Quality_Atmosphere_BRDF_Correction', 'MOD13Q1_061__250m_16_days_VI_Quality_Atmosphere_BRDF_Correction_Description', 'MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask', 'MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask_Description', 'MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice', 'MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice_Description']


In [9]:
ecija_raw_df = ecija_raw_df.drop(columns=ecija_constant)

In [10]:
ecija_raw_df.head()

,Date,MOD13Q1_061__250m_16_days_EVI,MOD13Q1_061__250m_16_days_NDVI,MOD13Q1_061__250m_16_days_VI_Quality,MOD13Q1_061__250m_16_days_pixel_reliability,MOD13Q1_061__250m_16_days_VI_Quality_bitmask,MOD13Q1_061__250m_16_days_VI_Quality_MODLAND,MOD13Q1_061__250m_16_days_VI_Quality_MODLAND_Description,MOD13Q1_061__250m_16_days_VI_Quality_VI_Usefulness,MOD13Q1_061__250m_16_days_VI_Quality_VI_Usefulness_Description,...,MOD13Q1_061__250m_16_days_VI_Quality_Aerosol_Quantity_Description,MOD13Q1_061__250m_16_days_VI_Quality_Adjacent_cloud_detected,MOD13Q1_061__250m_16_days_VI_Quality_Adjacent_cloud_detected_Description,MOD13Q1_061__250m_16_days_VI_Quality_Mixed_Clouds,MOD13Q1_061__250m_16_days_VI_Quality_Mixed_Clouds_Description,MOD13Q1_061__250m_16_days_VI_Quality_Possible_shadow,MOD13Q1_061__250m_16_days_VI_Quality_Possible_shadow_Description,MOD13Q1_061__250m_16_days_pixel_reliability_bitmask,MOD13Q1_061__250m_16_days_pixel_reliability_MODLAND,MOD13Q1_061__250m_16_days_pixel_reliability_MODLAND_Description
0,2000-02-18,0.3269,0.4989,2116.0,0.0,0b0000100001000100,0b00,VI produced with good quality,0b0001,Lower quality,...,Low,0b0,No,0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
1,2000-03-05,0.4503,0.5964,2116.0,0.0,0b0000100001000100,0b00,VI produced with good quality,0b0001,Lower quality,...,Low,0b0,No,0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
2,2000-03-21,0.4772,0.6279,2116.0,0.0,0b0000100001000100,0b00,VI produced with good quality,0b0001,Lower quality,...,Low,0b0,No,0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
3,2000-04-06,0.2790,0.5747,2116.0,0.0,0b0000100001000100,0b00,VI produced with good quality,0b0001,Lower quality,...,Low,0b0,No,0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
4,2000-04-22,0.2579,0.4957,2116.0,0.0,0b0000100001000100,0b00,VI produced with good quality,0b0001,Lower quality,...,Low,0b0,No,0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"


In [11]:
ecija_raw_df = ecija_raw_df.drop_duplicates()

In [12]:
ecija_raw_df.isna().sum().sum()

np.int64(0)

In [13]:
ecija_raw_df.describe()

,Date,MOD13Q1_061__250m_16_days_EVI,MOD13Q1_061__250m_16_days_NDVI,MOD13Q1_061__250m_16_days_VI_Quality,MOD13Q1_061__250m_16_days_pixel_reliability
count,614,614.000000,614.000000,614.000000,614.000000
mean,2013-06-17 16:32:03.127035,0.227895,0.351528,2236.618893,0.105863
min,2000-02-18 00:00:00,0.082000,0.155900,2062.000000,0.000000
25%,2006-10-20 00:00:00,0.126850,0.214925,2116.000000,0.000000
50%,2013-06-18 00:00:00,0.158500,0.264550,2116.000000,0.000000
75%,2020-02-14 00:00:00,0.271625,0.438875,2116.000000,0.000000
max,2026-10-16 00:00:00,0.865500,0.896600,35156.000000,3.000000
std,NaN,0.155809,0.190298,1878.227597,0.391839


In [14]:
keep_final_columns = ['Date', 'MOD13Q1_061__250m_16_days_EVI','MOD13Q1_061__250m_16_days_NDVI','MOD13Q1_061__250m_16_days_pixel_reliability']

ecija_processed_df = ecija_raw_df[keep_final_columns]

In [15]:
ecija_processed_df.to_csv(os.path.join(ndvi_processed_folder_loc,ecija_processed_file), index=False)

In [16]:
ndvi_processed_folder_loc = '../data/processed/ndvi/'

jerez_processed_file = 'jerez_ndvi_p.csv'
ubeda_processed_file = 'ubeda_ndvi_p.csv'

#### 4.2 Guadix dataset

In [17]:
guadix_raw_df = pd.read_csv(guadix_raw_file_loc)
guadix_df, guadix_report = quality_report(guadix_raw_df)

In [18]:
guadix_df.head()

,Category,ID,Latitude,Longitude,Date,MODIS_Tile,MOD13Q1_061_Line_Y_250m,MOD13Q1_061_Sample_X_250m,MOD13Q1_061__250m_16_days_EVI,MOD13Q1_061__250m_16_days_NDVI,...,MOD13Q1_061__250m_16_days_VI_Quality_Mixed_Clouds_Description,MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask,MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask_Description,MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice,MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice_Description,MOD13Q1_061__250m_16_days_VI_Quality_Possible_shadow,MOD13Q1_061__250m_16_days_VI_Quality_Possible_shadow_Description,MOD13Q1_061__250m_16_days_pixel_reliability_bitmask,MOD13Q1_061__250m_16_days_pixel_reliability_MODLAND,MOD13Q1_061__250m_16_days_pixel_reliability_MODLAND_Description
0,zone,guadix,37.24278,-3.16138,2000-02-18,h17v05,1323.0,3591.0,0.0924,0.1845,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
1,zone,guadix,37.24278,-3.16138,2000-03-05,h17v05,1323.0,3591.0,0.1025,0.1976,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
2,zone,guadix,37.24278,-3.16138,2000-03-21,h17v05,1323.0,3591.0,0.1130,0.2196,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
3,zone,guadix,37.24278,-3.16138,2000-04-06,h17v05,1323.0,3591.0,0.1299,0.2283,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
4,zone,guadix,37.24278,-3.16138,2000-04-22,h17v05,1323.0,3591.0,0.1477,0.2660,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"


In [19]:
print(guadix_report['constant_values'])

['Category', 'ID', 'Latitude', 'Longitude', 'MODIS_Tile', 'MOD13Q1_061_Line_Y_250m', 'MOD13Q1_061_Sample_X_250m', 'MOD13Q1_061__250m_16_days_VI_Quality_Aerosol_Quantity', 'MOD13Q1_061__250m_16_days_VI_Quality_Aerosol_Quantity_Description', 'MOD13Q1_061__250m_16_days_VI_Quality_Adjacent_cloud_detected', 'MOD13Q1_061__250m_16_days_VI_Quality_Adjacent_cloud_detected_Description', 'MOD13Q1_061__250m_16_days_VI_Quality_Atmosphere_BRDF_Correction', 'MOD13Q1_061__250m_16_days_VI_Quality_Atmosphere_BRDF_Correction_Description', 'MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask', 'MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask_Description', 'MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice', 'MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice_Description', 'MOD13Q1_061__250m_16_days_VI_Quality_Possible_shadow', 'MOD13Q1_061__250m_16_days_VI_Quality_Possible_shadow_Description']


In [20]:
guadix_report['NaN_values'].sum()

np.int64(0)

In [21]:
guadix_df.describe()

,Latitude,Longitude,MOD13Q1_061_Line_Y_250m,MOD13Q1_061_Sample_X_250m,MOD13Q1_061__250m_16_days_EVI,MOD13Q1_061__250m_16_days_NDVI,MOD13Q1_061__250m_16_days_VI_Quality,MOD13Q1_061__250m_16_days_pixel_reliability
count,614.00000,6.140000e+02,614.0,614.0,614.000000,614.000000,614.000000,614.000000
mean,37.24278,-3.161380e+00,1323.0,3591.0,0.121997,0.218143,2118.459283,0.011401
std,0.00000,4.444513e-16,0.0,0.0,0.073077,0.106764,41.892009,0.145179
min,37.24278,-3.161380e+00,1323.0,3591.0,0.033100,0.080200,2116.000000,0.000000
25%,37.24278,-3.161380e+00,1323.0,3591.0,0.075250,0.141425,2116.000000,0.000000
50%,37.24278,-3.161380e+00,1323.0,3591.0,0.095050,0.184850,2116.000000,0.000000
75%,37.24278,-3.161380e+00,1323.0,3591.0,0.145775,0.261600,2116.000000,0.000000
max,37.24278,-3.161380e+00,1323.0,3591.0,0.490200,0.688400,3154.000000,3.000000


In [22]:
guadix_processed_df = final_df(guadix_df)

In [23]:
guadix_processed_df.to_csv(os.path.join(ndvi_processed_folder_loc,guadix_processed_file), index=False)

#### 4.3 Jerez dataset

In [24]:
jerez_raw_df = pd.read_csv(jerez_raw_file_loc)
jerez_df, jerez_report = quality_report(jerez_raw_df)

In [25]:
jerez_df.head()

,Category,ID,Latitude,Longitude,Date,MODIS_Tile,MOD13Q1_061_Line_Y_250m,MOD13Q1_061_Sample_X_250m,MOD13Q1_061__250m_16_days_EVI,MOD13Q1_061__250m_16_days_NDVI,...,MOD13Q1_061__250m_16_days_VI_Quality_Mixed_Clouds_Description,MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask,MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask_Description,MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice,MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice_Description,MOD13Q1_061__250m_16_days_VI_Quality_Possible_shadow,MOD13Q1_061__250m_16_days_VI_Quality_Possible_shadow_Description,MOD13Q1_061__250m_16_days_pixel_reliability_bitmask,MOD13Q1_061__250m_16_days_pixel_reliability_MODLAND,MOD13Q1_061__250m_16_days_pixel_reliability_MODLAND_Description
0,zone,jerez,36.74074,-6.1837,2000-02-18,h17v05,1564.0,2421.0,0.1280,0.1739,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
1,zone,jerez,36.74074,-6.1837,2000-03-05,h17v05,1564.0,2421.0,0.1381,0.1857,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000001,0b00000000000000000000000000000001,"Marginal data, Useful, but look at other QA in..."
2,zone,jerez,36.74074,-6.1837,2000-03-21,h17v05,1564.0,2421.0,0.1409,0.1712,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000001,0b00000000000000000000000000000001,"Marginal data, Useful, but look at other QA in..."
3,zone,jerez,36.74074,-6.1837,2000-04-06,h17v05,1564.0,2421.0,0.1540,0.1752,...,Yes,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000011,0b00000000000000000000000000000011,"Cloudy data, Target not visible, covered with ..."
4,zone,jerez,36.74074,-6.1837,2000-04-22,h17v05,1564.0,2421.0,0.1645,0.2100,...,Yes,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000011,0b00000000000000000000000000000011,"Cloudy data, Target not visible, covered with ..."


In [26]:
print(jerez_report['constant_values'])

['Category', 'ID', 'Latitude', 'Longitude', 'MODIS_Tile', 'MOD13Q1_061_Line_Y_250m', 'MOD13Q1_061_Sample_X_250m', 'MOD13Q1_061__250m_16_days_VI_Quality_Atmosphere_BRDF_Correction', 'MOD13Q1_061__250m_16_days_VI_Quality_Atmosphere_BRDF_Correction_Description', 'MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask', 'MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask_Description', 'MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice', 'MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice_Description']


In [27]:
jerez_report['NaN_values'].sum()

np.int64(0)

In [28]:
jerez_df.describe()

,Latitude,Longitude,MOD13Q1_061_Line_Y_250m,MOD13Q1_061_Sample_X_250m,MOD13Q1_061__250m_16_days_EVI,MOD13Q1_061__250m_16_days_NDVI,MOD13Q1_061__250m_16_days_VI_Quality,MOD13Q1_061__250m_16_days_pixel_reliability
count,6.140000e+02,6.140000e+02,614.0,614.0,614.000000,614.000000,614.000000,614.000000
mean,3.674074e+01,-6.183700e+00,1564.0,2421.0,0.191256,0.248233,2622.711726,1.214984
std,7.111221e-15,1.777805e-15,0.0,0.0,0.053312,0.059227,3279.429223,1.248570
min,3.674074e+01,-6.183700e+00,1564.0,2421.0,0.045500,0.124100,2062.000000,0.000000
25%,3.674074e+01,-6.183700e+00,1564.0,2421.0,0.147650,0.202375,2118.000000,0.000000
50%,3.674074e+01,-6.183700e+00,1564.0,2421.0,0.184900,0.238700,2120.000000,1.000000
75%,3.674074e+01,-6.183700e+00,1564.0,2421.0,0.229125,0.286950,2124.000000,3.000000
max,3.674074e+01,-6.183700e+00,1564.0,2421.0,0.357300,0.449700,36134.000000,3.000000


In [29]:
jerez_processed_df = final_df(jerez_df)

In [30]:
jerez_processed_df.to_csv(os.path.join(ndvi_processed_folder_loc,jerez_processed_file), index=False)

#### 4. Úbeda dataset

In [31]:
ubeda_raw_df = pd.read_csv(ubeda_raw_file_loc)
ubeda_df, ubeda_report = quality_report(ubeda_raw_df)

In [32]:
ubeda_df.head()

,Category,ID,Latitude,Longitude,Date,MODIS_Tile,MOD13Q1_061_Line_Y_250m,MOD13Q1_061_Sample_X_250m,MOD13Q1_061__250m_16_days_EVI,MOD13Q1_061__250m_16_days_NDVI,...,MOD13Q1_061__250m_16_days_VI_Quality_Mixed_Clouds_Description,MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask,MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask_Description,MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice,MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice_Description,MOD13Q1_061__250m_16_days_VI_Quality_Possible_shadow,MOD13Q1_061__250m_16_days_VI_Quality_Possible_shadow_Description,MOD13Q1_061__250m_16_days_pixel_reliability_bitmask,MOD13Q1_061__250m_16_days_pixel_reliability_MODLAND,MOD13Q1_061__250m_16_days_pixel_reliability_MODLAND_Description
0,zone,ubeda,38.0072,-3.26722,2000-02-18,h17v05,956.0,3564.0,0.1854,0.3097,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
1,zone,ubeda,38.0072,-3.26722,2000-03-05,h17v05,956.0,3564.0,0.1859,0.3053,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
2,zone,ubeda,38.0072,-3.26722,2000-03-21,h17v05,956.0,3564.0,0.1945,0.3602,...,Yes,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000011,0b00000000000000000000000000000011,"Cloudy data, Target not visible, covered with ..."
3,zone,ubeda,38.0072,-3.26722,2000-04-06,h17v05,956.0,3564.0,0.2066,0.2960,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"
4,zone,ubeda,38.0072,-3.26722,2000-04-22,h17v05,956.0,3564.0,0.1912,0.3129,...,No,0b001,Land (Nothing else but land),0b0,No,0b0,No,0b00000000000000000000000000000000,0b00000000000000000000000000000000,"Good data, use with confidence"


In [33]:
print(ubeda_report['constant_values'])

['Category', 'ID', 'Latitude', 'Longitude', 'MODIS_Tile', 'MOD13Q1_061_Line_Y_250m', 'MOD13Q1_061_Sample_X_250m', 'MOD13Q1_061__250m_16_days_VI_Quality_Atmosphere_BRDF_Correction', 'MOD13Q1_061__250m_16_days_VI_Quality_Atmosphere_BRDF_Correction_Description', 'MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask', 'MOD13Q1_061__250m_16_days_VI_Quality_Land/Water_Mask_Description', 'MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice', 'MOD13Q1_061__250m_16_days_VI_Quality_Possible_snow/ice_Description']


In [34]:
ubeda_report['NaN_values'].sum()

np.int64(0)

In [35]:
ubeda_df.describe()

,Latitude,Longitude,MOD13Q1_061_Line_Y_250m,MOD13Q1_061_Sample_X_250m,MOD13Q1_061__250m_16_days_EVI,MOD13Q1_061__250m_16_days_NDVI,MOD13Q1_061__250m_16_days_VI_Quality,MOD13Q1_061__250m_16_days_pixel_reliability
count,6.140000e+02,6.140000e+02,614.0,614.0,614.000000,614.000000,614.000000,614.000000
mean,3.800720e+01,-3.267220e+00,956.0,3564.0,0.239602,0.363979,2333.226384,0.214984
std,7.111221e-15,8.889026e-16,0.0,0.0,0.040257,0.079257,2301.491398,0.715275
min,3.800720e+01,-3.267220e+00,956.0,3564.0,0.074600,0.228000,2062.000000,0.000000
25%,3.800720e+01,-3.267220e+00,956.0,3564.0,0.211625,0.294925,2116.000000,0.000000
50%,3.800720e+01,-3.267220e+00,956.0,3564.0,0.232550,0.351900,2116.000000,0.000000
75%,3.800720e+01,-3.267220e+00,956.0,3564.0,0.259150,0.422400,2120.000000,0.000000
max,3.800720e+01,-3.267220e+00,956.0,3564.0,0.453100,0.631500,35152.000000,3.000000


In [36]:
ubeda_processed_df = final_df(ubeda_df)

In [37]:
ubeda_processed_df.to_csv(os.path.join(ndvi_processed_folder_loc,ubeda_processed_file), index=False)